In [1]:
%%capture
!pip install facenet-pytorch

## Libraries

In [2]:
import sys
sys.path.append('/home/pj00/projects/Github/small_face_recognition_trcking/utils')

In [64]:
import numpy as np
import cv2
import os
import time
import torch
import torch.nn as nn
from tqdm import tqdm
import torchvision
from PIL import Image
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training
from custom_model import distill_model, distill_model_2
from torch.utils.data import Subset, DataLoader
from image_iter import FaceDataset, customSubset
from torch.utils.tensorboard import SummaryWriter
from torch import optim
from torch.optim.lr_scheduler import MultiStepLR


from utils import extract_face, take_picture, detect_crop_image, next_folder_name, aug_img, create_neg_class, model_size

%matplotlib inline
import matplotlib.pyplot as plt

## Functions

In [105]:
def create_class_data(data_folder, num_classes, num_images, aug_transform, dataset, idx_dict, model, crop_transform, device):
    class_path = []

    if num_classes == 1:
        path = os.path.join(data_folder, '{}'.format(next_folder_name(data_folder)))
        class_path.append(path)
        try:
            os.mkdir(path)
        except:
            print('Path exists: Issue!')
        create_neg_class(path, num_images, aug_transform, dataset, idx_dict) # add 50 random images

        frame, save_path = extract_face(save_path=data_folder)
        class_path.append(save_path)
        save_path = os.path.join(save_path, '0.jpg')
        images = detect_crop_image(frame=frame, model=mtcnn, transform=transform, device=device)
        pil_image = to_pil_image(images[0].detach().cpu())
        pil_image.save(save_path)
        aug_img(save_path=save_path, transform=aug_transform, num_images=num_images)

    else:
        pass
    
    return class_path

In [106]:
def save_model(model, model_name, optimizer_name, epoch, model_weight_path):
    os.makedirs(model_weight_path, exist_ok=True)  # Create ckpt directory if it doesn't exist
    file_name = f"{model_name}_{optimizer_name}_{epoch}.pt"
    save_path = os.path.join(model_weight_path, file_name)
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to {save_path}")


## Constants

In [109]:
BATCH_SIZE=32
data_path = '/home/pj00/projects/Github/small_face_recognition_trcking/Data/finetuning_data'
train_root = '/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec'
num_classes = 1
num_images = 2048
epochs = 8

## Models

In [78]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [79]:
mtcnn = MTCNN(
    image_size=112,
    margin=0,
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=False,
    device=device
)

In [80]:
resnet = InceptionResnetV1(
    classify=True,
    pretrained='casia-webface',
    num_classes=2
).to(device)

In [154]:
student = distill_model().to(device)
weight_path = '/home/pj00/projects/Github/small_face_recognition_trcking/model-weights/mobV3_adam_29.pt'
student.load_state_dict(torch.load(weight_path))

<All keys matched successfully>

In [155]:
# for param in student.parameters():
#     param.requires_grad = False

In [165]:
student_class = nn.Sequential(student, nn.Hardswish(), nn.Dropout(p=0.2, inplace=True), nn.Linear(512, 2))

In [166]:
# student.final = nn.Linear(512, 2)

In [167]:
student_class.to(device)
print('Done')
# for name, param in student_class.named_parameters():
#     if param.requires_grad:
#         print(name)

Done


In [168]:
model_size(student_class)

model size: 65397824 / bit | 8.17 / MB


## Transforms

In [99]:
transform = transforms.Compose([
            transforms.Resize((112, 112)),
            transforms.ToTensor(),
        ])

In [100]:
aug_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=.5, hue=.3),
    transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.)), 
    transforms.RandomAutocontrast(),
    transforms.ToPILImage()])

In [101]:
train_dataset_transform = transforms.Compose([
    transforms.ToTensor(),
])

## Create Dataset

In [17]:
dataset = FaceDataset(path_imgrec=train_root, rand_mirror=True)

/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec /home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.idx
header0 label [490624. 501196.]
id2range 10572


In [18]:
ss = customSubset(train_root)
idx_dict = ss.generate_idx_dic()

In [108]:
class_paths = create_class_data(data_folder=data_path, 
                                num_classes=num_classes, 
                                num_images=num_images, 
                                aug_transform=aug_transform, 
                                dataset=dataset, 
                                idx_dict=idx_dict, 
                                model=mtcnn, 
                                crop_transform=transform, 
                                device=device)

[ WARN:0@2268.597] global /croot/opencv-suite_1691620365762/work/modules/videoio/src/cap_gstreamer.cpp (862) isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created


Photo taken!


In [110]:
train_dataset = torchvision.datasets.ImageFolder(
    root='/home/pj00/projects/Github/small_face_recognition_trcking/Data/finetuning_data',
    transform=train_dataset_transform
)

In [111]:
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True)

## Finetuning

In [112]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.parameters(), lr=0.001)
scheduler = MultiStepLR(optimizer, [5, 10])

### Finetuning Resnet

In [113]:
model_weight_path = '/home/pj00/projects/Github/small_face_recognition_trcking/model-weights/finetuned/resnet'

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.parameters(), lr=0.001)
scheduler = MultiStepLR(optimizer, [5, 10])

In [114]:
losses = []
for epoch in tqdm(range(epochs)):
    epoch_loss = 0
    for x, y in train_loader:
        resnet.train()
        x = x.to(device)
        y = y.to(device)
        y_pred = resnet(x)
        loss_batch = loss_fn(y_pred, y)
        optimizer.zero_grad()
        loss_batch.backward()
        optimizer.step()

        epoch_loss += loss_batch.item()

    epoch_loss /= len(train_loader)
    losses.append(epoch_loss)
    print('Epoch_loss: {}'.format(epoch_loss))
    
    scheduler.step()
    
    save_model(resnet, model_name='resnet', optimizer_name='adam', epoch=str(epoch+1), model_weight_path=model_weight_path)

 12%|█████▋                                       | 1/8 [00:28<03:20, 28.68s/it]

Epoch_loss: 0.009807941974603551
Model saved to /home/pj00/projects/Github/small_face_recognition_trcking/model-weights/finetuned/resnet/resnet_adam_1.pt


 25%|███████████▎                                 | 2/8 [00:57<02:52, 28.80s/it]

Epoch_loss: 5.9362615513425254e-05
Model saved to /home/pj00/projects/Github/small_face_recognition_trcking/model-weights/finetuned/resnet/resnet_adam_2.pt
Epoch_loss: 1.426237957380394e-05


 38%|████████████████▉                            | 3/8 [01:33<02:40, 32.12s/it]

...

Model saved to /home/pj00/projects/Github/small_face_recognition_trcking/model-weights/finetuned/resnet/resnet_adam_6.pt
Epoch_loss: 6.454401386468689e-06


 88%|███████████████████████████████████████▍     | 7/8 [04:07<00:37, 37.05s/it]

Model saved to /home/pj00/projects/Github/small_face_recognition_trcking/model-weights/finetuned/resnet/resnet_adam_7.pt
Epoch_loss: 8.464803787033848e-05


100%|█████████████████████████████████████████████| 8/8 [04:46<00:00, 35.80s/it]

Model saved to /home/pj00/projects/Github/small_face_recognition_trcking/model-weights/finetuned/resnet/resnet_adam_8.pt


### Finetuned MobileNetV3

In [169]:
model_weight_path = '/home/pj00/projects/Github/small_face_recognition_trcking/model-weights/finetuned/mobV3'

In [170]:
optimizer = optim.SGD(student_class.parameters(), lr=0.01, momentum=0.9)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5, verbose=True)

In [171]:
losses = []
for epoch in tqdm(range(epochs)):
    epoch_loss = 0
    for x, y in train_loader:
        student_class.train()
        x = x.to(device)
        y = y.to(device)
        y_pred = student_class(x)
        loss_batch = loss_fn(y_pred, y)
        optimizer.zero_grad()
        loss_batch.backward()
        optimizer.step()

        epoch_loss += loss_batch.item()

    epoch_loss /= len(train_loader)
    losses.append(epoch_loss)
    print('Epoch_loss: {}'.format(epoch_loss))
    
#     scheduler.step()
    
    save_model(student_class, model_name='mobV3', optimizer_name='adam', epoch=str(epoch+1), model_weight_path=model_weight_path)

 12%|█████▋                                       | 1/8 [00:06<00:42,  6.12s/it]

Epoch_loss: 0.05157690908708901
Model saved to /home/pj00/projects/Github/small_face_recognition_trcking/model-weights/finetuned/mobV3/mobV3_adam_1.pt


 25%|███████████▎                                 | 2/8 [00:12<00:36,  6.11s/it]

Epoch_loss: 0.004458231841454108
Model saved to /home/pj00/projects/Github/small_face_recognition_trcking/model-weights/finetuned/mobV3/mobV3_adam_2.pt


 38%|████████████████▉                            | 3/8 [00:18<00:30,  6.11s/it]

...

Epoch_loss: 0.0007797421064879018
Model saved to /home/pj00/projects/Github/small_face_recognition_trcking/model-weights/finetuned/mobV3/mobV3_adam_6.pt


 88%|███████████████████████████████████████▍     | 7/8 [00:43<00:06,  6.24s/it]

Epoch_loss: 0.000716238964173499
Model saved to /home/pj00/projects/Github/small_face_recognition_trcking/model-weights/finetuned/mobV3/mobV3_adam_7.pt


100%|█████████████████████████████████████████████| 8/8 [00:49<00:00,  6.20s/it]

Epoch_loss: 0.0005469522019438955
Model saved to /home/pj00/projects/Github/small_face_recognition_trcking/model-weights/finetuned/mobV3/mobV3_adam_8.pt
